In [1]:
import numpy as np
import pandas
import sklearn 
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from mlxtend.plotting import plot_decision_regions
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn import datasets
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
#import the data
data = pandas.read_csv('./Project.csv', delimiter=',')

In [ ]:
#print the data
data

,user_id,day_type,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,burnout_score,burnout_risk
0,1,Weekday,9.59,11.86,4,2,0,7.55,91.2,19.17,Low
1,1,Weekend,7.38,10.33,4,1,0,6.69,82.0,29.70,Low
2,1,Weekend,6.31,8.92,1,2,0,8.87,80.6,32.93,Low
3,1,Weekday,8.34,10.70,4,1,1,8.13,70.0,45.47,Low
4,1,Weekend,6.97,9.83,1,2,0,5.85,67.1,51.61,Low
...,...,...,...,...,...,...,...,...,...,...,...
1795,180,Weekend,6.33,8.16,0,4,0,5.59,73.5,31.91,Low
1796,180,Weekend,4.70,7.88,0,4,0,6.69,89.8,26.30,Low
1797,180,Weekend,3.92,6.39,2,1,0,6.77,74.6,34.07,Low
1798,180,Weekday,8.93,11.11,2,5,0,8.28,74.6,38.14,Low


In [ ]:
#set variables
hours_worked = data["work_hours"]
screen_time = data["screen_time_hours"]
meetings_count = data["meetings_count"]
breaks_taken = data["breaks_taken"]
after_hours_work = data["after_hours_work"]
sleep_hours = data["sleep_hours"]
task_completion = data["task_completion_rate"]
burnout_score = data["burnout_score"]
burnout_risk = data["burnout_risk"]

In [5]:
x = data.drop(columns = ['user_id','burnout_score'])
y = data['burnout_score']

In [6]:
x_nominal = pandas.get_dummies(x)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=500)
xn_train, xn_test, yn_train, yn_test = train_test_split(x_nominal, y, test_size=500)

labels = ['model'] + x_nominal.columns.to_list()
labels
spearman_table = pandas.DataFrame(columns = labels)
spearman_table

,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium


## Linear

### Linear Regression

In [8]:
linear = LinearRegression()

In [11]:
linear.fit(xn_train, yn_train)
y_predicted_lr = linear.predict(xn_test)
mean_squared_error(yn_test, y_predicted_lr)

30.326424727975127

In [12]:
y_predicted_lr = pandas.DataFrame(y_predicted_lr)

In [13]:
vals = {}
vals['model'] = 'linear regression'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_predicted_lr, method='spearman')

spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

### Ridge Regression

## Decision Tree

### Regression Tree

In [14]:
tree = DecisionTreeRegressor()
parameters_tree = {# 'criterion':('squared_error','friedman_mse','absolute_error','poisson'),
                   'max_depth':[5,10,15,20],
                   'min_samples_split':[5,10,15,20],
                   'max_leaf_nodes':[5,10,20,50,None]
                  }
clf = GridSearchCV(tree, parameters_tree)
clf.fit(xn_train, yn_train)

GridSearchCV(estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': [5, 10, 15, 20],
                         'max_leaf_nodes': [5, 10, 20, 50, None],
                         'min_samples_split': [5, 10, 15, 20]})

In [15]:
y_predicted_rt = pandas.DataFrame(clf.predict(xn_test))

vals = {}
vals['model'] = 'regression tree'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_predicted_rt, method='spearman')
    
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

In [16]:
spearman_table

,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium
0,linear regression,-0.016633,-0.033408,-0.009921,0.022236,0.003316,-0.078257,0.003675,-0.022407,0.022407,-0.012871,0.025758,-0.022178
0,regression tree,-0.016073,-0.031928,-0.005310,0.016911,0.001299,-0.077914,0.004429,-0.023459,0.023459,-0.017090,0.026874,-0.021775


In [17]:
x_nominal.columns

Index(['work_hours', 'screen_time_hours', 'meetings_count', 'breaks_taken',
       'after_hours_work', 'sleep_hours', 'task_completion_rate',
       'day_type_Weekday', 'day_type_Weekend', 'burnout_risk_High',
       'burnout_risk_Low', 'burnout_risk_Medium'],
      dtype='object')

### Random Forest

In [18]:
rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
rf_reg.fit(xn_train, yn_train)
y_pred_rf = pandas.DataFrame(rf_reg.predict(xn_test))

vals = {}
vals['model'] = 'random forest'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_pred_rf, method='spearman')
    
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

## Boosting

### Gradient Boosting

In [19]:
gb_reg = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)
gb_reg.fit(xn_train, yn_train)
y_pred_gb = pandas.DataFrame(gb_reg.predict(xn_test))

vals = {}
vals['model'] = 'gradient boosting'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_pred_gb, method='spearman')

spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

In [20]:
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])
spearman_table


,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium
0,linear regression,-0.016633,-0.033408,-0.009921,0.022236,0.003316,-0.078257,0.003675,-0.022407,0.022407,-0.012871,0.025758,-0.022178
0,regression tree,-0.016073,-0.031928,-0.005310,0.016911,0.001299,-0.077914,0.004429,-0.023459,0.023459,-0.017090,0.026874,-0.021775
0,random forest,-0.019747,-0.037332,-0.015325,0.016957,0.006760,-0.079245,-0.004666,-0.026094,0.026094,-0.013497,0.014714,-0.010381
0,gradient boosting,-0.015207,-0.032598,-0.005433,0.018077,0.004639,-0.076146,0.000132,-0.021493,0.021493,-0.017614,0.023887,-0.018452
0,gradient boosting,-0.015207,-0.032598,-0.005433,0.018077,0.004639,-0.076146,0.000132,-0.021493,0.021493,-0.017614,0.023887,-0.018452
